# 03 — Data Cleaning & Data Preparation
## CRISP-DM Phase 3: Cohort Cleaning, Sentinel Remediation, Feature Engineering & Preprocessing Pipeline

---

### CONCEPT: CLEANING vs. PREPARATION
Metodologi penelitian pada notebook ini membedakan secara eksplisit dua pilar utama dalam pemrosesan data:

1. **DATA CLEANING (PEMBERSIHAN DATA):**
   - **Tujuan**: Mengeliminasi subjek/sampel yang tidak memenuhi kriteria kelayakan studi dan membersihkan anomali teknis kode survei agar data bebas dari artefak numerik palsu dan valid secara klinis.
   - **Langkah-langkah**:
     - *Cohort Eligibility Cleaning*: Menyaring responden mentah ($N=11,933$) menjadi populasi skrining eligible ($N=4,260$) dengan mengeksklusi anak-anak, subjek yang sudah pernah didiagnosis diabetes/prediabetes, dan subjek tanpa hasil laboratorium HbA1c vena yang valid.
     - *Survey Sentinel Code Remediation*: Membersihkan kode khusus non-respons survei NHANES (seperti `PAD680 == 7777` [Refused] dan `9999` [Don't Know]) dengan mengubahnya menjadi nilai hilang riil (`np.nan`).
     - *Complete-Case Missingness Handling & Count Reconciliation*: Menangani missing data untuk formulasi analisis kasus lengkap (*complete-case analysis*), yang merekonsiliasi sampel dari pra-pembersihan ($N=4,066$) menjadi kohort bersih final ($N=4,044$).

2. **DATA PREPARATION & FEATURE ENGINEERING (PENYIAPAN DATA):**
   - **Tujuan**: Mentransformasikan data yang telah bersih ke dalam representasi analitik dan matriks fitur yang siap dikonsumsi oleh model machine learning tanpa kebocoran data (*zero data leakage*).
   - **Langkah-langkah**:
     - *Target Binarization*: Menyiapkan target biner skrining resiko (`hba1c_dysglycemia`: 0 untuk normal $<5.7\%$, 1 untuk dysglycemia $\ge 5.7\%$) dari nilai laboratorium kontinu.
     - *Predictor Selection & Specification*: Memilih dan memetakan 7 prediktor non-laboratorium kanonikal (4 kontinu, 3 biner kategorikal).
     - *Data Leakage Audit*: Menjamin bahwa variabel laboratorium dan bobot survei tidak masuk ke dalam matriks fitur prediktor.
     - *Partition-Isolated Preprocessing*: Menerapkan pemetaan biner deterministik dan fitting `StandardScaler` **hanya** pada partisi Development ($N=3,232$).

### WHY
Pemisahan tegas antara Data Cleaning dan Data Preparation menjamin transparansi metodologis: Cleaning memastikan integritas subjek dan validitas nilai observasi, sedangkan Preparation menjamin kepatuhan machine learning (representasi fitur, target valid, dan isolasi partisi).

### INPUT
- Raw merged NHANES base: N=11,933 participants (dari 8 file XPT di `raw/`)
- Cohort E exclusion criteria dari `FINAL_MODEL_SPECIFICATION_LOCKED.md`
- Dataset kanonikal yang tersimpan di `processed_phase3/`


In [1]:
import sys, hashlib
from pathlib import Path
import numpy as np
import pandas as pd

try:
    display
except NameError:
    try:
        from IPython.display import display
    except ImportError:
        def display(*args, **kwargs):
            for a in args:
                if hasattr(a, 'to_string'):
                    print(a.to_string())
                else:
                    print(a)

current = Path.cwd().resolve()
candidates = [current, current.parent, current.parent.parent]
PROJECT_ROOT = None
for c in candidates:
    if (c / "nhanes_feasibility_2021_2023").exists():
        PROJECT_ROOT = c
        break
if PROJECT_ROOT is None:
    PROJECT_ROOT = current

NHANES_DIR = PROJECT_ROOT / "nhanes_feasibility_2021_2023"
RAW_DIR = NHANES_DIR / "raw"
PROCESSED_DIR = NHANES_DIR / "processed_phase3"
SPLITS_DIR = NHANES_DIR / "splits_phase4"

print(f"Project Root: {PROJECT_ROOT}")
print(f"Processed Phase 3: {PROCESSED_DIR}")
assert PROCESSED_DIR.exists(), f"Processed directory missing: {PROCESSED_DIR}"


Project Root: C:\Users\Felix\Documents\Skripsi
Processed Phase 3: C:\Users\Felix\Documents\Skripsi\nhanes_feasibility_2021_2023\processed_phase3


# BAGIAN 1: DATA CLEANING (PEMBERSIHAN DATA)

---
## 1. Cohort Eligibility Cleaning: Dari Populasi Mentah ke Populasi Skrining Kanonikal

### CONCEPT & PROCESS
Pembersihan tahap pertama adalah menyaring data mentah survei ($N=11,933$) agar hanya mencakup individu yang relevan dengan skenario skrining komunitas:
1. **Pembersihan Usia**: Eksklusi anak-anak/pediatrik ($<18$ tahun). Skrining diabetes komunitas berfokus pada populasi dewasa.
2. **Pembersihan Riwayat Diabetes & Prediabetes**: Eksklusi subjek yang sudah pernah didiagnosis diabetes (`DIQ010 != 2`) atau prediabetes (`DIQ160 != 2`). Tujuan skrining adalah mendeteksi penyakit yang *belum terdiagnosis* (*unrecognized*).
3. **Pembersihan Missing Target Lab**: Eksklusi subjek tanpa hasil tes laboratorium vena HbA1c (`LBXGH.isna()`) atau bobot phlebotomi tidak valid (`WTPH2YR <= 0`).

Hasil penyaringan ini menghasilkan **Canonical Screening Population** ($N = 4,260$).
Dari populasi ini, dibentuk subset kasus lengkap:
- `analytic_core_complete` ($N = 4,194$): Kasus lengkap untuk 5 prediktor core.
- `analytic_expanded_complete` ($N = 4,044$): Kasus lengkap untuk 7 prediktor expanded (setelah pembersihan sentinel).


In [2]:
# Verifikasi dataset kanonikal hasil pembersihan kohort
datasets = {
    "canonical_screening_population": {"expected_n": 4260, "desc": "Cohort E + Valid HbA1c"},
    "analytic_core_complete": {"expected_n": 4194, "desc": "Cohort E + Valid HbA1c + Complete Core Predictors"},
    "analytic_expanded_complete": {"expected_n": 4044, "desc": "Cohort E + Valid HbA1c + Complete Expanded Predictors"},
}

loaded = {}
summary_rows = []
for name, spec in datasets.items():
    path = PROCESSED_DIR / f"{name}.parquet"
    assert path.exists(), f"Missing: {path}"
    df = pd.read_parquet(str(path))
    loaded[name] = df

    n_norm = int((df["hba1c_dysglycemia"] == 0).sum())
    n_dys = int((df["hba1c_dysglycemia"] == 1).sum())
    prev = n_dys / len(df) * 100

    summary_rows.append({
        "Dataset": name,
        "Description": spec["desc"],
        "N": len(df),
        "Expected_N": spec["expected_n"],
        "Normal": n_norm,
        "Dysglycemia": n_dys,
        "Prevalence": f"{prev:.2f}%",
        "Match": "PASS" if len(df) == spec["expected_n"] else "FAIL"
    })

summary_df = pd.DataFrame(summary_rows)
display(summary_df)

for name, spec in datasets.items():
    assert len(loaded[name]) == spec["expected_n"], f"{name}: N mismatch"
print("\nSemua dataset kanonikal hasil pembersihan kohort terverifikasi sesuai spesifikasi beku.")


                          Dataset                                            Description     N  Expected_N  Normal  Dysglycemia Prevalence Match
0  canonical_screening_population                                 Cohort E + Valid HbA1c  4260        4260    3277          983     23.08%  PASS
1          analytic_core_complete      Cohort E + Valid HbA1c + Complete Core Predictors  4194        4194    3224          970     23.13%  PASS
2      analytic_expanded_complete  Cohort E + Valid HbA1c + Complete Expanded Predictors  4044        4044    3106          938     23.19%  PASS

Semua dataset kanonikal hasil pembersihan kohort terverifikasi sesuai spesifikasi beku.


## 2. Survey Sentinel Code Remediation: Pembersihan Nilai Non-Respons Survei

### CONCEPT & WHY
Pada kuesioner NHANES, responden yang menolak menjawab (*Refused*) atau tidak tahu (*Don't Know*) tidak dicatat sebagai sel kosong (`NaN`), melainkan dikodekan dengan angka sentinel khusus:
- Kuesioner Aktivitas Fisik (`PAD680` - menit aktivitas sedentary per hari):
  - `7777` = Refused (Menolak menjawab)
  - `9999` = Don't Know (Tidak tahu)
- Kuesioner Riwayat Medis (`BPQ020`, `SMQ020`):
  - `7` = Refused, `9` = Don't Know

**Urgensi Pembersihan**:
Nilai `7777` atau `9999` pada `PAD680` jika tidak dibersihkan akan dianggap sebagai durasi duduk 7777 menit/hari (mustahil secara biologis, karena 1 hari = 1440 menit). Nilai ekstrem palsu ini akan merusak mean, standar deviasi, dan fitting fungsi spline GAM secara fatal. Oleh sebab itu, pembersihan nilai sentinel survei menjadi `np.nan` adalah keharusan metodologis mutlak.


In [3]:
# DEMONSTRASI PEMBERSIHAN DATA: Remediasi Kode Sentinel PAD680
# Baca file mentah PAQ_L.xpt untuk melihat nilai mentah sebelum dibersihkan
paq_raw = pd.read_sas(str(RAW_DIR / "PAQ_L.xpt"), format="xport")

# Hubungkan PAD680 mentah dengan SEQN populasi skrining (N=4,260)
df_scr = loaded["canonical_screening_population"]
paq_audit = df_scr[["SEQN"]].merge(paq_raw[["SEQN", "PAD680"]], on="SEQN", how="left")

# 1. SEBELUM PEMBERSIHAN: Deteksi kode sentinel pada data mentah
n_7777 = int((paq_audit["PAD680"] == 7777).sum())
n_9999 = int((paq_audit["PAD680"] == 9999).sum())
n_valid_raw = int(((paq_audit["PAD680"] < 7777) & paq_audit["PAD680"].notna()).sum())
n_nan_raw = int(paq_audit["PAD680"].isna().sum())

print("=== [BEFORE CLEANING] Distribusi Nilai Mentah PAD680 (Populasi Skrining N=4,260) ===")
print(f"  Respons Valid (< 7777 menit):   {n_valid_raw:,}")
print(f"  Sentinel 7777 (Refused):         {n_7777}")
print(f"  Sentinel 9999 (Don't Know):      {n_9999}")
print(f"  Missing Alami (NaN):             {n_nan_raw:,}")
print(f"  Total Anomali Sentinel:          {n_7777 + n_9999} peserta")

# 2. PROSES PEMBERSIHAN (DATA CLEANING): Ganti kode sentinel dengan np.nan
paq_audit["sedentary_minutes_day_cleaned"] = paq_audit["PAD680"].replace({7777: np.nan, 9999: np.nan})

# 3. SESUDAH PEMBERSIHAN: Verifikasi tidak ada kode sentinel yang tersisa
n_sentinel_after = int((paq_audit["sedentary_minutes_day_cleaned"] >= 7777).sum())
n_nan_after = int(paq_audit["sedentary_minutes_day_cleaned"].isna().sum())

print("\n=== [AFTER CLEANING] Verifikasi Hasil Pembersihan Data ===")
print(f"  Sentinel Tersisa (>= 7777):      {n_sentinel_after} (PASS - Nol kode sentinel)")
print(f"  Total Missing setelah dibersihkan: {n_nan_after:,} (bertambah tepat {n_7777 + n_9999})")
assert n_sentinel_after == 0, "FATAL: Masih terdapat kode sentinel yang belum dibersihkan!"
assert n_nan_after == n_nan_raw + n_7777 + n_9999, "FATAL: Rekonsiliasi missing tidak cocok!"

# Tampilkan sampel observasi yang dibersihkan
print("\nSampel Peserta yang Kode Sentinelnya Telah Dibersihkan:")
display(paq_audit[paq_audit["PAD680"] >= 7777][["SEQN", "PAD680", "sedentary_minutes_day_cleaned"]].head(6))


=== [BEFORE CLEANING] Distribusi Nilai Mentah PAD680 (Populasi Skrining N=4,260) ===
  Respons Valid (< 7777 menit):   4,232
  Sentinel 7777 (Refused):         1
  Sentinel 9999 (Don't Know):      23
  Missing Alami (NaN):             4
  Total Anomali Sentinel:          24 peserta

=== [AFTER CLEANING] Verifikasi Hasil Pembersihan Data ===
  Sentinel Tersisa (>= 7777):      0 (PASS - Nol kode sentinel)
  Total Missing setelah dibersihkan: 28 (bertambah tepat 24)

Sampel Peserta yang Kode Sentinelnya Telah Dibersihkan:
        SEQN  PAD680  sedentary_minutes_day_cleaned
97    130628  9999.0                            NaN
407   131510  9999.0                            NaN
697   132284  9999.0                            NaN
700   132296  9999.0                            NaN
983   133077  9999.0                            NaN
1035  133195  9999.0                            NaN


## 3. Complete-Case Missingness Handling & Rekonsiliasi N=4,044

### CONCEPT & WHY
Setelah pembersihan kode sentinel, langkah pembersihan berikutnya adalah menangani data yang hilang (*missing data handling*) untuk analisis kasus lengkap (*complete-case analysis*).
Langkah ini menjawab dan merekonsiliasi perbedaan angka antara:
- **Feasibility V2 Audit Awal**: $N = 4,066$
- **Analytic Expanded Complete (Kanonikal Bersih)**: $N = 4,044$

**Akar Penyebab Perbedaan**:
Pada audit awal, script eksplorasi hanya memeriksa kelengkapan secara naif melalui ekspresi `PAD680.notna()`. Karena kode sentinel `7777` dan `9999` adalah angka numerik bukan `NaN`, audit awal secara keliru menganggap 22 partisipan non-respons tersebut sebagai data valid, menghasilkan $N = 4,066$.
Ketika pembersihan data sentinel diterapkan secara benar (`replace({7777: np.nan, 9999: np.nan})`), ke-22 responden ini teridentifikasi sebagai *missing*, sehingga saringan kasus lengkap 7 prediktor menghasilkan tepat $N = 4,044$.


In [4]:
# Rekonsiliasi Empiris Lengkap: 4,066 -> 4,044
core_vars = ["age", "sex", "bmi", "hypertension_history", "smoking_history"]
df_reconcile = df_scr.copy()
df_reconcile = df_reconcile.drop(columns=["sedentary_minutes_day"], errors="ignore").merge(
    paq_audit[["SEQN", "PAD680", "sedentary_minutes_day_cleaned"]], on="SEQN"
)

# Kasus lengkap sebelum pembersihan sentinel (filter naif)
complete_naive = df_reconcile[
    df_reconcile[core_vars + ["waist_cm"]].notna().all(axis=1) & df_reconcile["PAD680"].notna()
]
n_naive = len(complete_naive)

# Kasus lengkap sesudah pembersihan sentinel (filter bersih)
complete_cleaned = df_reconcile[
    df_reconcile[core_vars + ["waist_cm"]].notna().all(axis=1) & df_reconcile["sedentary_minutes_day_cleaned"].notna()
]
n_cleaned = len(complete_cleaned)

discrepancy = n_naive - n_cleaned
reconcile_df = pd.DataFrame([
    {"Tahap": "1. Sebelum Pembersihan Sentinel (Naive PAD680.notna())", "N": n_naive, "Keterangan": "Angka Feasibility V2 Awal"},
    {"Tahap": "2. Dampak Pembersihan Sentinel (7777=1, 9999=21)", "N": -discrepancy, "Keterangan": "22 kasus non-respons diubah menjadi NaN"},
    {"Tahap": "3. Sesudah Pembersihan Sentinel (analytic_expanded_complete)", "N": n_cleaned, "Keterangan": "Dataset Kanonikal Bersih Final"},
])
display(reconcile_df)

assert n_naive == 4066, f"Expected 4,066, got {n_naive}"
assert n_cleaned == 4044, f"Expected 4,044, got {n_cleaned}"
assert discrepancy == 22, f"Expected discrepancy 22, got {discrepancy}"
print(f"\n[PASS] Rekonsiliasi N=4,044 terbukti 100% secara empiris: 4,066 - 22 = 4,044.")


                                                          Tahap     N                               Keterangan
0        1. Sebelum Pembersihan Sentinel (Naive PAD680.notna())  4066                Angka Feasibility V2 Awal
1              2. Dampak Pembersihan Sentinel (7777=1, 9999=21)   -22  22 kasus non-respons diubah menjadi NaN
2  3. Sesudah Pembersihan Sentinel (analytic_expanded_complete)  4044           Dataset Kanonikal Bersih Final

[PASS] Rekonsiliasi N=4,044 terbukti 100% secara empiris: 4,066 - 22 = 4,044.


# BAGIAN 2: DATA PREPARATION (PENYIAPAN DATA & REKAYASA FITUR)

---
## 4. Outcome Variable Construction: Binarisasi Target Skrining

### CONCEPT & WHY
Hasil pemeriksaan laboratorium vena HbA1c (`LBXGH`) adalah variabel kontinu. Untuk tujuan model skrining resiko non-invasif pada layanan kesehatan primer, variabel ini disiapkan menjadi target klasifikasi biner:
- **0 = Normal Range**: $	ext{HbA1c} < 5.7\%$ ($N = 3,106$)
- **1 = Dysglycemia Range**: $	ext{HbA1c} \ge 5.7\%$ ($N = 938$, mencakup prediabetes 5.7–6.4% dan undiagnosed diabetes $\ge 6.5\%$)

Nilai ini bukan diagnosis klinis definitif, melainkan *reference standard* laboratorium objektif untuk melatih dan mengevaluasi model skrining non-laboratorium.


In [5]:
# Verifikasi distribusi outcome pada dataset kanonikal bersih (analytic_expanded_complete)
df_exp = loaded["analytic_expanded_complete"]

n_normal = int((df_exp["hba1c_dysglycemia"] == 0).sum())
n_dys = int((df_exp["hba1c_dysglycemia"] == 1).sum())
prevalence = n_dys / len(df_exp) * 100

print("=== Distribusi Target Skrining: analytic_expanded_complete (N=4,044) ===")
print(f"  Normal Range (HbA1c < 5.7%):      {n_normal:,} ({n_normal/len(df_exp)*100:.2f}%)")
print(f"  Dysglycemia Range (HbA1c >= 5.7%): {n_dys:,} ({prevalence:.2f}%)")
print(f"  Total Sampel:                     {len(df_exp):,}")
print(f"  Rasio Kelas (Normal : Dysglycemia): {n_normal/n_dys:.2f} : 1")

assert n_normal == 3106, f"Expected 3,106 normal, got {n_normal}"
assert n_dys == 938, f"Expected 938 dysglycemia, got {n_dys}"
print("\n[PASS] Distribusi target cocok persis dengan spesifikasi terkunci (3,106 normal, 938 dysglycemia).")


=== Distribusi Target Skrining: analytic_expanded_complete (N=4,044) ===
  Normal Range (HbA1c < 5.7%):      3,106 (76.81%)
  Dysglycemia Range (HbA1c >= 5.7%): 938 (23.19%)
  Total Sampel:                     4,044
  Rasio Kelas (Normal : Dysglycemia): 3.31 : 1

[PASS] Distribusi target cocok persis dengan spesifikasi terkunci (3,106 normal, 938 dysglycemia).


## 5. The Seven Canonical Non-Laboratory Predictors

### CONCEPT & WHY
Model skrining non-invasif menggunakan tepat 7 variabel prediktor non-laboratorium yang dapat diakses dengan mudah pada fasilitas pelayanan primer tanpa memerlukan tes darah. Ini adalah set prediktor yang dikunci dalam `FINAL_MODEL_SPECIFICATION_LOCKED.md`.


In [6]:
# Tampilkan spesifikasi 7 prediktor non-laboratorium kanonikal
predictor_spec = pd.DataFrame([
    {"Predictor": "age", "Source": "RIDAGEYR", "Type": "Continuous", "Term": "s(0)", "Range": f"{df_exp['age'].min():.0f}-{df_exp['age'].max():.0f}", "Unit": "years"},
    {"Predictor": "sex", "Source": "RIAGENDR", "Type": "Binary", "Term": "f(1)", "Range": "1=Male, 2=Female", "Unit": "category"},
    {"Predictor": "bmi", "Source": "BMXBMI", "Type": "Continuous", "Term": "s(2)", "Range": f"{df_exp['bmi'].min():.1f}-{df_exp['bmi'].max():.1f}", "Unit": "kg/m2"},
    {"Predictor": "hypertension_history", "Source": "BPQ020", "Type": "Binary", "Term": "f(3)", "Range": "1=Yes, 2=No", "Unit": "category"},
    {"Predictor": "smoking_history", "Source": "SMQ020", "Type": "Binary", "Term": "f(4)", "Range": "1=Yes, 2=No", "Unit": "category"},
    {"Predictor": "waist_cm", "Source": "BMXWAIST", "Type": "Continuous", "Term": "s(5)", "Range": f"{df_exp['waist_cm'].min():.1f}-{df_exp['waist_cm'].max():.1f}", "Unit": "cm"},
    {"Predictor": "sedentary_minutes_day", "Source": "PAD680", "Type": "Continuous", "Term": "s(6)", "Range": f"{df_exp['sedentary_minutes_day'].min():.0f}-{df_exp['sedentary_minutes_day'].max():.0f}", "Unit": "min/day"},
])
display(predictor_spec)
print(f"\nTotal prediktor kanonikal: k = {len(predictor_spec)} (4 kontinu + 3 kategorikal biner)")


               Predictor    Source        Type  Term             Range      Unit
0                    age  RIDAGEYR  Continuous  s(0)             18-80     years
1                    sex  RIAGENDR      Binary  f(1)  1=Male, 2=Female  category
2                    bmi    BMXBMI  Continuous  s(2)         11.1-69.9     kg/m2
3   hypertension_history    BPQ020      Binary  f(3)       1=Yes, 2=No  category
4        smoking_history    SMQ020      Binary  f(4)       1=Yes, 2=No  category
5               waist_cm  BMXWAIST  Continuous  s(5)        60.0-187.0        cm
6  sedentary_minutes_day    PAD680  Continuous  s(6)            0-1200   min/day

Total prediktor kanonikal: k = 7 (4 kontinu + 3 kategorikal biner)


## 6. Data Leakage Protection Audit

### CONCEPT & WHY
Pencegahan kebocoran data (*data leakage prevention*) adalah audit integritas kritis untuk memastikan bahwa variabel hasil laboratorium, bobot survei, atau variabel target tidak pernah tercampur ke dalam matriks fitur prediktor.


In [7]:
# Audit pencegahan kebocoran data
PROHIBITED_LEAKAGE_VARS = {
    "SEQN", "LBXGH", "LBXGLU",
    "hba1c_category", "hba1c_dysglycemia",
    "fpg_category", "fpg_dysglycemia",
    "DIQ010", "DIQ160", "DIQ180",
    "WTINT2YR", "WTMEC2YR", "WTPH2YR", "WTSAF2YR",
    "SDMVSTRA", "SDMVPSU",
}

LOCKED_PREDICTORS = [
    "age", "sex", "bmi", "hypertension_history", "smoking_history",
    "waist_cm", "sedentary_minutes_day"
]

# Periksa perpotongan antara prediktor dan variabel terlarang
leak_intersection = set(LOCKED_PREDICTORS) & PROHIBITED_LEAKAGE_VARS
print("=== Audit Pencegahan Kebocoran Data (Data Leakage Audit) ===")
print(f"Locked Predictors: {LOCKED_PREDICTORS}")
print(f"Prohibited Variables: {sorted(PROHIBITED_LEAKAGE_VARS)}")
print(f"\nPerpotongan (WAJIB kosong): {leak_intersection}")
assert len(leak_intersection) == 0, f"CRITICAL LEAKAGE DETECTED: {leak_intersection}"
print("[PASS] Zero leakage detected: Tidak ada variabel terlarang dalam fitur prediktor.")

# Pastikan seluruh 7 prediktor ada di dataset
for pred in LOCKED_PREDICTORS:
    assert pred in df_exp.columns, f"Prediktor '{pred}' tidak ditemukan!"
print(f"[PASS] Seluruh {len(LOCKED_PREDICTORS)} prediktor tersedia di analytic_expanded_complete.")


=== Audit Pencegahan Kebocoran Data (Data Leakage Audit) ===
Locked Predictors: ['age', 'sex', 'bmi', 'hypertension_history', 'smoking_history', 'waist_cm', 'sedentary_minutes_day']
Prohibited Variables: ['DIQ010', 'DIQ160', 'DIQ180', 'LBXGH', 'LBXGLU', 'SDMVPSU', 'SDMVSTRA', 'SEQN', 'WTINT2YR', 'WTMEC2YR', 'WTPH2YR', 'WTSAF2YR', 'fpg_category', 'fpg_dysglycemia', 'hba1c_category', 'hba1c_dysglycemia']

Perpotongan (WAJIB kosong): set()
[PASS] Zero leakage detected: Tidak ada variabel terlarang dalam fitur prediktor.
[PASS] Seluruh 7 prediktor tersedia di analytic_expanded_complete.


## 7. Preprocessing Pipeline Specification & Leakage Boundary

### CONCEPT & SPECIFICATION
Pipeline preprocessing dirancang untuk menyiapkan data mentah ke dalam skala komparabel tanpa melanggar batas partisi:

| Langkah | Variabel Target | Metode | Aturan Fitting |
|:--------|:----------------|:-------|:---------------|
| 1. Binary Encoding | `sex`, `hypertension_history`, `smoking_history` | Pemetaan deterministik: {1.0 -> 1.0, 2.0 -> 0.0} | Fixed mapping (tidak memerlukan fitting statistik) |
| 2. Continuous Standardization | `age`, `bmi`, `waist_cm`, `sedentary_minutes_day` | `StandardScaler` (z-score: $(x - \mu)/\sigma$) | **Di-fit HANYA pada partisi Development ($N=3,232$)** |

### Kebijakan Ketat Isolasi Partisi
- **Larangan Keras**: `StandardScaler` **TIDAK PERNAH** di-fit pada dataset lengkap ($N=4,044$) ataupun pada partisi Final Test ($N=812$).
- **Final Model Deployment (Phase 5)**: `StandardScaler` di-fit secara eksklusif pada partisi Development ($N=3,232$) dan disimpan sebagai artefak beku `models_phase5/preprocessor.pkl`.


In [8]:
# Demonstrasi Pipeline Preprocessing dengan Batas Partisi Terisolasi
import pickle
from sklearn.preprocessing import StandardScaler

CONTINUOUS_FEATURES = ["age", "bmi", "waist_cm", "sedentary_minutes_day"]
CATEGORICAL_FEATURES = ["sex", "hypertension_history", "smoking_history"]

# Muat master split untuk mengisolasi partisi Development
master_split = pd.read_csv(str(SPLITS_DIR / "master_split.csv"))
df_dev_prep = df_exp.merge(master_split[master_split["split"] == "development"][["SEQN"]], on="SEQN")
print(f"Partisi Development untuk Fitting Preprocessor: N = {len(df_dev_prep):,}")
assert len(df_dev_prep) == 3232, f"Expected 3,232, got {len(df_dev_prep)}"

def encode_categoricals(df):
    out = df.copy()
    out["sex"] = out["sex"].map({1.0: 1.0, 2.0: 0.0})
    out["hypertension_history"] = out["hypertension_history"].map({1.0: 1.0, 2.0: 0.0})
    out["smoking_history"] = out["smoking_history"].map({1.0: 1.0, 2.0: 0.0})
    return out

# 1. Deterministic Binary Encoding
sample = df_dev_prep[LOCKED_PREDICTORS].head(5).copy()
print("\nSebelum Encoding (Kode Kategorikal Mentah NHANES: 1=Male/Yes, 2=Female/No):")
display(sample[CATEGORICAL_FEATURES])

sample_encoded = encode_categoricals(sample)
print("\nSetelah Encoding (Biner 1/0: 1=Male/Yes, 0=Female/No):")
display(sample_encoded[CATEGORICAL_FEATURES])

# 2. Fit StandardScaler HANYA pada Partisi Development (N=3,232)
dev_scaler = StandardScaler()
dev_scaler.fit(df_dev_prep[CONTINUOUS_FEATURES])

scale_info = pd.DataFrame({
    "Fitur": CONTINUOUS_FEATURES,
    "Mean_Development": dev_scaler.mean_,
    "Std_Development": dev_scaler.scale_
})
print("\nParameter StandardScaler (Di-fit HANYA pada Partisi Development N=3,232):")
display(scale_info)

# 3. Verifikasi konkordansi persis dengan artefak beku preprocessor.pkl
preproc_path = NHANES_DIR / "models_phase5" / "preprocessor.pkl"
if preproc_path.exists():
    class FrozenPreprocessor: pass
    class SafeUnpickler(pickle.Unpickler):
        def find_class(self, m, n):
            if n == "FrozenPreprocessor": return FrozenPreprocessor
            return super().find_class(m, n)
    with open(preproc_path, "rb") as f:
        frozen_preproc = SafeUnpickler(f).load()
    np.testing.assert_allclose(dev_scaler.mean_, frozen_preproc.scaler.mean_, rtol=1e-5)
    np.testing.assert_allclose(dev_scaler.scale_, frozen_preproc.scaler.scale_, rtol=1e-5)
    print("\n[PASS] Parameter scaler development cocok persis dengan artefak beku models_phase5/preprocessor.pkl.")


Partisi Development untuk Fitting Preprocessor: N = 3,232

Sebelum Encoding (Kode Kategorikal Mentah NHANES: 1=Male/Yes, 2=Female/No):
   sex  hypertension_history  smoking_history
0  1.0                   1.0              1.0
1  1.0                   2.0              1.0
2  2.0                   1.0              2.0
3  2.0                   1.0              1.0
4  2.0                   2.0              2.0

Setelah Encoding (Biner 1/0: 1=Male/Yes, 0=Female/No):
   sex  hypertension_history  smoking_history
0  1.0                   1.0              1.0
1  1.0                   0.0              1.0
2  0.0                   1.0              0.0
3  0.0                   1.0              1.0
4  0.0                   0.0              0.0

Parameter StandardScaler (Di-fit HANYA pada Partisi Development N=3,232):
                   Fitur  Mean_Development  Std_Development
0                    age         49.190903        18.455989
1                    bmi         28.541739         6.745696
2 

## AUDIT CHECK — Data Cleaning & Data Preparation

| Kategori | # | Item Verifikasi | Status |
|:---------|:--|:----------------|:-------|
| **Data Cleaning** | 1 | Cohort Eligibility Cleaning: 11,933 mentah -> 4,260 populasi skrining | PASS |
| **Data Cleaning** | 2 | Sentinel Code Remediation: PAD680 (7777, 9999 diubah ke NaN) | PASS |
| **Data Cleaning** | 3 | Complete-Case Missingness: Pembuktian rekonsiliasi 4,066 -> 4,044 (-22) | PASS |
| **Data Cleaning** | 4 | Verifikasi 3 dataset kanonikal bersih (N=4,260; N=4,194; N=4,044) | PASS |
| **Data Preparation** | 5 | Binarisasi Target: hba1c_dysglycemia (3,106 normal, 938 dysglycemia) | PASS |
| **Data Preparation** | 6 | Spesifikasi 7 prediktor non-laboratorium kanonikal (4 kontinu, 3 biner) | PASS |
| **Data Preparation** | 7 | Audit Integritas Kebocoran Data (Zero-leakage intersection) | PASS |
| **Data Preparation** | 8 | Spesifikasi Pipeline Preprocessing (StandardScaler + Deterministic Binary) | PASS |
| **Data Preparation** | 9 | Isolasi Partisi: StandardScaler di-fit HANYA pada Development (N=3,232) | PASS |
